In [1]:
# ============================================================
# CABITZA ET AL. COVID-19 MODEL REPRODUCTION
# RANDOM FOREST — COMPLETE PIPELINE
# ============================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)
from sklearn.impute import KNNImputer
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    brier_score_loss
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. PROJECT SETTINGS
# ------------------------------------------------------------

RANDOM_STATE = 42

PROJECT_ROOT = Path(
    r"C:\Users\bl\Documents\ML-Projects\covid-laboratory-prediction"
)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "all_training.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "reproduction"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("CABITZA ET AL. — RANDOM FOREST REPRODUCTION")
print("=" * 65)
print("Dataset:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

id_col = "Unnamed: 0"
target_col = "target"

ids = df[id_col].astype(str).str.strip()


# ------------------------------------------------------------
# 3. RECONSTRUCT COHORTS
# ------------------------------------------------------------

def assign_cohort(patient_id):
    if patient_id.startswith("A"):
        return "training"
    elif patient_id.startswith("PSMAY"):
        return "training"
    elif patient_id.isdigit() and 2001 <= int(patient_id) <= 2058:
        return "internal_external"
    elif patient_id.isdigit():
        return "external"
    else:
        return "unclassified"


df["cohort"] = ids.apply(assign_cohort)

print("\nCohort reconstruction:")
print(df["cohort"].value_counts())


# ------------------------------------------------------------
# 4. DEFINE COVID-SPECIFIC DEVELOPMENT DATASET
# ------------------------------------------------------------

predictors = [
    col for col in df.columns
    if col not in [id_col, target_col, "cohort"]
]

development_df = df[
    df["cohort"] == "training"
].copy()

X = development_df[predictors].copy()
y = development_df[target_col].astype(int).copy()

print("\nDevelopment dataset")
print("-------------------")
print("N =", len(development_df))
print("Predictors =", len(predictors))
print("\nOutcome:")
print(y.value_counts().sort_index())


# ------------------------------------------------------------
# 5. SAME 80:20 STRATIFIED SPLIT AS LOGISTIC REGRESSION
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nData split")
print("-------------------")
print("Training:", X_train.shape)
print(y_train.value_counts().sort_index())

print("\nHold-out:", X_test.shape)
print(y_test.value_counts().sort_index())


# ------------------------------------------------------------
# 6. RANDOM FOREST ESTIMATORS
# ------------------------------------------------------------

# Separate RF estimator for RFE
rf_selector = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight=None
)

# Final classifier
rf_classifier = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight=None
)


# ------------------------------------------------------------
# 7. PIPELINE
#
# Cabitza et al. reported:
#   KNN imputation (k=5)
#   normalization
#   RFE feature selection
#   classifier
#
# Scaling is not included here because Random Forest is invariant
# to monotonic feature scaling and does not require standardized
# predictors.
# ------------------------------------------------------------

pipeline = Pipeline([
    (
        "imputer",
        KNNImputer(n_neighbors=5)
    ),
    (
        "rfe",
        RFE(
            estimator=rf_selector,
            step=1
        )
    ),
    (
        "classifier",
        rf_classifier
    )
])


# ------------------------------------------------------------
# 8. HYPERPARAMETER GRID
#
# Original Supplementary Table 1 grid is unavailable.
# Therefore this is an independently specified reproduction grid.
# ------------------------------------------------------------

param_grid = {
    "rfe__n_features_to_select": [
        10,
        20,
        30,
        34
    ],

    "classifier__n_estimators": [
        100,
        300
    ],

    "classifier__max_depth": [
        None,
        10
    ],

    "classifier__min_samples_split": [
        2,
        5
    ]
}


# ------------------------------------------------------------
# 9. 5-FOLD STRATIFIED CROSS-VALIDATION
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

n_candidates = (
    len(param_grid["rfe__n_features_to_select"])
    * len(param_grid["classifier__n_estimators"])
    * len(param_grid["classifier__max_depth"])
    * len(param_grid["classifier__min_samples_split"])
)

print("\nGrid search")
print("-------------------")
print("Candidates:", n_candidates)
print("CV folds: 5")
print("Total fits:", n_candidates * 5)
print("\nStarting Random Forest grid search...")

start_time = time.time()

grid_search.fit(
    X_train,
    y_train
)

elapsed = time.time() - start_time

print("\nGrid search completed.")
print(f"Elapsed time: {elapsed / 60:.2f} minutes")


# ------------------------------------------------------------
# 10. BEST CROSS-VALIDATED MODEL
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BEST RANDOM FOREST MODEL")
print("=" * 65)

print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

print("\nBest hyperparameters:")
for parameter, value in grid_search.best_params_.items():
    print(f"{parameter}: {value}")


# ------------------------------------------------------------
# 11. TOP CROSS-VALIDATION RESULTS
# ------------------------------------------------------------

cv_results = pd.DataFrame(
    grid_search.cv_results_
)

cv_summary = cv_results[
    [
        "param_rfe__n_features_to_select",
        "param_classifier__n_estimators",
        "param_classifier__max_depth",
        "param_classifier__min_samples_split",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
].copy()

cv_summary = cv_summary.sort_values(
    "mean_test_score",
    ascending=False
)

print("\nTop 10 CV configurations:")
print(
    cv_summary.head(10).to_string(index=False)
)


# ------------------------------------------------------------
# 12. EVALUATE ON UNTOUCHED HOLD-OUT SET
# ------------------------------------------------------------

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(
    y_test,
    y_pred
)

auc = roc_auc_score(
    y_test,
    y_prob
)

brier = brier_score_loss(
    y_test,
    y_prob
)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred
).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)


# ------------------------------------------------------------
# 13. DISPLAY HOLD-OUT PERFORMANCE
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("HOLD-OUT PERFORMANCE")
print("=" * 65)

print(f"ROC-AUC:     {auc:.4f}")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Brier score: {brier:.4f}")

print("\nConfusion matrix:")
print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")


# ------------------------------------------------------------
# 14. SELECTED FEATURES
# ------------------------------------------------------------

rfe = best_rf.named_steps["rfe"]

selected_features = np.array(
    predictors
)[rfe.support_]

print("\nSelected features:")
print(f"Number selected: {len(selected_features)}")

for feature in selected_features:
    print(feature)


# ------------------------------------------------------------
# 15. FEATURE IMPORTANCE
# ------------------------------------------------------------

final_rf = best_rf.named_steps["classifier"]

feature_importance = pd.DataFrame({
    "feature": selected_features,
    "importance": final_rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("\nTop Random Forest feature importances:")
print(
    feature_importance.head(15).to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 16. CREATE RESULT TABLE
# ------------------------------------------------------------

rf_results = pd.DataFrame({
    "model": [
        "Random Forest"
    ],
    "cv_auc": [
        grid_search.best_score_
    ],
    "holdout_auc": [
        auc
    ],
    "accuracy": [
        accuracy
    ],
    "sensitivity": [
        sensitivity
    ],
    "specificity": [
        specificity
    ],
    "brier_score": [
        brier
    ],
    "true_negative": [
        tn
    ],
    "false_positive": [
        fp
    ],
    "false_negative": [
        fn
    ],
    "true_positive": [
        tp
    ],
    "n_train": [
        len(X_train)
    ],
    "n_holdout": [
        len(X_test)
    ],
    "n_features_rfe": [
        len(selected_features)
    ],
    "best_n_estimators": [
        grid_search.best_params_[
            "classifier__n_estimators"
        ]
    ],
    "best_max_depth": [
        str(
            grid_search.best_params_[
                "classifier__max_depth"
            ]
        )
    ],
    "best_min_samples_split": [
        grid_search.best_params_[
            "classifier__min_samples_split"
        ]
    ]
})


# ------------------------------------------------------------
# 17. SAVE RESULTS
# ------------------------------------------------------------

rf_results.to_csv(
    RESULTS_DIR /
    "random_forest_holdout_results.csv",
    index=False
)

cv_summary.to_csv(
    RESULTS_DIR /
    "random_forest_cv_results.csv",
    index=False
)

feature_importance.to_csv(
    RESULTS_DIR /
    "random_forest_feature_importance.csv",
    index=False
)


# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL RANDOM FOREST REPRODUCTION RESULT")
print("=" * 65)

print(
    rf_results.round(4).to_string(
        index=False
    )
)

print("\nFiles saved to:")
print(RESULTS_DIR)

print("\nSaved files:")
print("- random_forest_holdout_results.csv")
print("- random_forest_cv_results.csv")
print("- random_forest_feature_importance.csv")

print("\nRandom Forest reproduction complete.")

CABITZA ET AL. — RANDOM FOREST REPRODUCTION
Dataset: C:\Users\bl\Documents\ML-Projects\covid-laboratory-prediction\data\raw\all_training.csv
Dataset exists: True

Cohort reconstruction:
cohort
training             1624
internal_external      58
external               54
Name: count, dtype: int64

Development dataset
-------------------
N = 1624
Predictors = 34

Outcome:
target
0    838
1    786
Name: count, dtype: int64

Data split
-------------------
Training: (1299, 34)
target
0    670
1    629
Name: count, dtype: int64

Hold-out: (325, 34)
target
0    168
1    157
Name: count, dtype: int64

Grid search
-------------------
Candidates: 32
CV folds: 5
Total fits: 160

Starting Random Forest grid search...
Fitting 5 folds for each of 32 candidates, totalling 160 fits

Grid search completed.
Elapsed time: 6.37 minutes

BEST RANDOM FOREST MODEL
Best CV ROC-AUC: 0.8885

Best hyperparameters:
classifier__max_depth: None
classifier__min_samples_split: 5
classifier__n_estimators: 300
rfe__n_f

In [2]:
# ============================================================
# PUBLISHED VS REPRODUCED RANDOM FOREST
# Cabitza et al. — COVID-specific dataset, standard model
# ============================================================

rf_comparison = pd.DataFrame({
    "metric": [
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "ROC-AUC",
        "Brier score"
    ],
    "published_cabitza": [
        0.84,
        0.84,
        0.84,
        0.84,
        0.12
    ],
    "reproduced": [
        accuracy,
        sensitivity,
        specificity,
        auc,
        brier
    ]
})

rf_comparison["difference"] = (
    rf_comparison["reproduced"]
    - rf_comparison["published_cabitza"]
)

print("=" * 65)
print("CABITZA RANDOM FOREST: PUBLISHED VS REPRODUCED")
print("=" * 65)

display(
    rf_comparison.round(4)
)

rf_comparison.to_csv(
    RESULTS_DIR /
    "random_forest_published_vs_reproduced.csv",
    index=False
)

print(
    "\nSaved:",
    RESULTS_DIR /
    "random_forest_published_vs_reproduced.csv"
)

CABITZA RANDOM FOREST: PUBLISHED VS REPRODUCED


,metric,published_cabitza,reproduced,difference
0,Accuracy,0.84,0.8062,-0.0338
1,Sensitivity,0.84,0.7962,-0.0438
2,Specificity,0.84,0.8155,-0.0245
3,ROC-AUC,0.84,0.8809,0.0409
4,Brier score,0.12,0.1411,0.0211



Saved: C:\Users\bl\Documents\ML-Projects\covid-laboratory-prediction\results\reproduction\random_forest_published_vs_reproduced.csv
